# PART A: Pytrends timeseries extraction

### Time spent: 2 hours

### Notes: 
- #### `get_historical_interest` function appears to be discontinued: https://github.com/GeneralMills/pytrends/pull/542
    - Using daily data from `interest_over_time` function instead (BUT it is daily not hourly)
- #### Google seems to have tightened restrictions: (successful solutions outlined here no longer work: https://github.com/GeneralMills/pytrends/pull/553)
    - Current solution: Insert random waiting times between requests and add error handling to catch when requests are blocked and then sleep for 60 seconds to reset api call limit (not perfect solution)
    - I would need rotating proxies or browser automation with Playwright to reliably reliably use pytrends api since it is unofficial
- A couple of the films with special characters were triggering 400 error response from the api - Just put placeholder null values for these films because of time limit but easy solution with a bit more time is to remove those characters in the `get_timeseries` function before executing `pytrends.build_payload`


In [302]:
import pandas as pd 
import numpy as np 
import pytrends
import matplotlib.pyplot as plt
from pytrends.request import TrendReq
import time
from sklearn.linear_model import LinearRegression
import pickle
from datetime import datetime, timedelta
import random
from scipy.stats import skew
pytrends = TrendReq(hl="en-US", tz=360)

df = pd.read_csv('releases.csv')
dd = pd.read_csv('demand_data.csv')


In [255]:
def get_timeseries(title, timeframe):

    pytrends.build_payload([title], timeframe=timeframe)
    iot = pytrends.interest_over_time()
    return iot

def get_21_days_before_after(title, release_date):
    previous_21 = datetime.strptime(release_date, "%Y-%m-%d") - timedelta(days=21)
    next_21 = datetime.strptime(release_date, "%Y-%m-%d") + timedelta(days=21)
    timeframe = str(previous_21)[:10] + ' ' + str(next_21)[:10]

    return get_timeseries(title, timeframe)
    
def get_all_time_series(titles, release_dates):
    collected_timeseries = {}
    consecutive_failures = 0

    queue = list(zip(titles, release_dates))

    while len(queue) > 0:
        title, release_date = queue.pop(0)

        try:
            collected_timeseries[title] = get_21_days_before_after(title, release_date)
            consecutive_failures = 0
            print(title)
            time.sleep(random.uniform(2, 5))

        except Exception as e:
            print(f"Error for {title}: {e}")
            consecutive_failures += 1

            if consecutive_failures >= 2:
                print("API failed consecutively. Breaking.")
                break

            print("Sleep for 60 seconds..")
            time.sleep(61)

            queue.append((title, release_date))  # put to the back of the queue 

    return collected_timeseries

In [256]:
all_timeseries = get_all_time_series(df.title.values, df.release_date.values)

Blue Beetle
Gran Turismo
The Nun II
Meg 2: The Trench
Retribution
Talk to Me
Fast X
Sound of Freedom
Barbie
Elemental
Operation Napoleon
Error for No One Will Save You: HTTPSConnectionPool(host='trends.google.com', port=443): Max retries exceeded with url: /trends/api/explore?hl=en-US&tz=360&req=%7B%22comparisonItem%22%3A+%5B%7B%22keyword%22%3A+%22No+One+Will+Save+You%22%2C+%22time%22%3A+%222023-09-01+2023-10-13%22%2C+%22geo%22%3A+%22%22%7D%5D%2C+%22category%22%3A+0%2C+%22property%22%3A+%22%22%7D (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x11ef52610>: Failed to resolve 'trends.google.com' ([Errno 8] nodename nor servname provided, or not known)"))
Sleep for 60 seconds..
Saw X
Carl's Date
The Black Book
Let Her Kill You
Kandahar
Transformers: Rise of the Beasts
Spy Kids: Armageddon
Indiana Jones and the Dial of Destiny
Til Death Do Us Part
Spider-Man: Across the Spider-Verse
The Creator
Teenage Mutant Ninja Turtles: Mutant Mayhem
Expend4bles
John Wick

#### Save to pickle for easy access down the line

In [260]:
with open("movies_interest_over_time.pkl", "wb") as f:
    pickle.dump(all_timeseries, f)

In [286]:
with open("movies_interest_over_time.pkl", "rb") as f:
    timeseries_pickle = pickle.load(f)
timeseries_pickle_df = pd.DataFrame({"title":timeseries_pickle.keys(), "timeseries_df":timeseries_pickle.values()})

### Trend Features asked for

- `trends_peak_pre_release`
Maximum value in the window [-21d, 0d] before release.

- `trends_peak_post_release`
Maximum value in the window [0d, +21d] after release.

- `trends_auc_pre_release`
Area under the curve (AUC) before release.

- `trends_slope_pre_release`
Trend slope during the last 7 days before release.

- `trends_release_day_value` 
Trend value on the release day.

- `trends_volatility_pre_release`
Standard deviation of trend values before release.


In [324]:
df_with_timeseries = df.merge(timeseries_pickle_df, how="inner", on="title")

max_pre_release = []
max_post_release = []
auc_pre_release = []
slope_7_days_pre_release = []
release_day_interest = []
stddev_interest_pre_release = []

for i in df_with_timeseries.title.values:
    # catch edge cases where pytrends returns empty array (such as with Operation Napoleon)
    if len(df_with_timeseries.loc[df_with_timeseries['title'] == i].timeseries_df.values[0]) > 0:

        ts = df_with_timeseries.loc[df_with_timeseries['title'] == i].timeseries_df.values[0][i].values

        # max pre release 
        max_pre_release.append(ts[:21].max())

        # max post release
        max_post_release.append(ts[22:].max())

        # auc pre release
        auc_pre_release.append(np.trapezoid(ts[:21]))

        # slope 7 days pre release
        y = ts[14:21]
        x = np.array(range(len(y))).reshape(-1, 1)
        model = LinearRegression()
        model.fit(x, y)
        slope_7_days_pre_release.append(model.coef_[0])

        # release day interest
        release_day_interest.append(ts[21])

        #std dev of interest pre release
        stddev_interest_pre_release.append(np.std(ts[:21]))

    else:
        max_pre_release.append(None)
        max_post_release.append(None)
        auc_pre_release.append(None)
        slope_7_days_pre_release.append(None)
        release_day_interest.append(None)
        stddev_interest_pre_release.append(None)


df_with_timeseries['max_pre_release'] = max_pre_release
df_with_timeseries['max_post_release'] = max_post_release
df_with_timeseries['auc_pre_release'] = auc_pre_release
df_with_timeseries['slope_7_days_pre_release'] = slope_7_days_pre_release
df_with_timeseries['release_day_interest'] = release_day_interest
df_with_timeseries['stddev_interest_pre_release'] = stddev_interest_pre_release

### Additional Trend Features that could be useful (engineered features focus more on before the release but numbers suggest post release date will also be relevant)

- `trend_skew`
Measure skew of trend either side of release date

- `peak_trend_day_position`
How many dates later or earlier was the "100" day recorded in trends

- `auc_post_peak` 
Area under the curve (AUC) after the "100" day recorded in trends

- `auc_post_release` 
Area under the curve (AUC) after the release date

- `trends_slope_post_release`
Trend slope during the first 7 days after release.

- `trends_slope_post_peak`
Trend slope during the first 7 days after peak in trends.

- `trends_day_7_value` 
Trend value on day 7 after release

In [325]:
trend_skew = []
peak_trend_day_position = []
auc_post_peak = []
auc_post_release = []
trends_slope_post_release = []
trends_slope_post_peak = []
trends_day_7_value = []


for i in df_with_timeseries.title.values:
    # catch edge cases where pytrends returns empty array (such as with Operation Napoleon)
    if len(df_with_timeseries.loc[df_with_timeseries['title'] == i].timeseries_df.values[0]) > 0:

        ts = df_with_timeseries.loc[df_with_timeseries['title'] == i].timeseries_df.values[0][i].values

        # trend_skew
        trend_skew.append(skew(ts))

        # peak_trend_day_position
        peak_trend_day_position.append(np.where(ts == 100)[0][0] - 21)

        # auc_post_peak
        auc_post_peak.append(np.trapezoid(ts[np.where(ts == 100)[0][0]:]))

        # auc_post_release
        auc_post_release.append(np.trapezoid(ts[21:]))

        # trends_slope_post_release
        y = ts[21:28]
        x = np.array(range(len(y))).reshape(-1, 1)
        model = LinearRegression()
        model.fit(x, y)
        trends_slope_post_release.append(model.coef_[0])

        # trends_slope_post_peak
        y = ts[np.where(ts == 100)[0][0]:np.where(ts == 100)[0][0] + 7]
        x = np.array(range(len(y))).reshape(-1, 1)
        model = LinearRegression()
        model.fit(x, y)
        trends_slope_post_peak.append(model.coef_[0])

        # trends_day_7_value
        trends_day_7_value.append(ts[28])


    else:
        trend_skew.append(None)
        peak_trend_day_position.append(None)
        auc_post_peak.append(None)
        auc_post_release.append(None)
        trends_slope_post_release.append(None)
        trends_slope_post_peak.append(None)
        trends_day_7_value.append(None)


df_with_timeseries['trend_skew'] = trend_skew
df_with_timeseries['peak_trend_day_position'] = peak_trend_day_position
df_with_timeseries['auc_post_peak'] = auc_post_peak
df_with_timeseries['auc_post_release'] = auc_post_release
df_with_timeseries['trends_slope_post_peak'] = trends_slope_post_peak
df_with_timeseries['trends_day_7_value'] = trends_day_7_value

## Final dataset

In [335]:
# include movies that failed the trends extraction (this is a temporary fix -
    # to make those movies work, i would just need to remove special characters
    # that were triggering 400 error response from the api - easy solution with a bit more time)
df_with_timeseries_final = df.merge(df_with_timeseries, on=['title', 'release_date', 'genres', 'overview', 'spoken_languages'], how="left")
df_with_timeseries_final = df_with_timeseries_final.drop("timeseries_df", axis=1)


In [339]:
# export to .csv
df_with_timeseries_final.set_index("title").to_csv("releases_with_timeseries_features.csv")

In [340]:
# read .csv
pd.read_csv("releases_with_timeseries_features.csv")

,title,release_date,genres,overview,spoken_languages,max_pre_release,max_post_release,auc_pre_release,slope_7_days_pre_release,release_day_interest,stddev_interest_pre_release,trend_skew,peak_trend_day_position,auc_post_peak,auc_post_release,trends_slope_post_peak,trends_day_7_value
0,Blue Beetle,2023-08-16,"Action, Science Fiction, Adventure",Recent college grad Jaime Reyes returns home f...,"English, Portuguese, Spanish",18.0,100.0,191.0,0.857143,34.0,3.681787,1.583642,3.0,696.5,913.5,-12.071429,43.0
1,Gran Turismo,2023-08-09,"Action, Drama, Adventure",The ultimate wish-fulfillment tale of a teenag...,"English, German, Japanese",26.0,100.0,330.5,1.750000,36.0,4.370588,1.331222,18.0,200.0,989.0,-17.500000,33.0
2,The Nun II,2023-09-06,"Horror, Mystery, Thriller","In 1956 France, a priest is violently murdered...","English, French",9.0,100.0,42.0,0.964286,18.0,2.223356,2.569714,2.0,462.5,556.5,-13.250000,24.0
3,Meg 2: The Trench,2023-08-02,"Action, Science Fiction, Horror",An exploratory dive into the deepest depths of...,English,18.0,100.0,155.5,1.535714,29.0,4.081372,1.992407,3.0,561.0,784.5,-11.821429,32.0
4,Retribution,2023-08-23,"Action, Mystery, Thriller, Crime",When a mysterious caller puts a bomb under his...,"English, German",18.0,100.0,233.5,0.964286,25.0,2.158147,2.396520,4.0,523.0,733.5,-9.428571,30.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268,Huesera: The Bone Woman,2023-02-10,"Drama, Horror, Mystery",Valeria's joy at becoming a first-time mother ...,Spanish,0.0,100.0,0.0,0.000000,42.0,0.000000,1.302997,9.0,458.5,749.5,-3.964286,73.0
269,Nefarious,2023-04-14,"Horror, Thriller","On the day of his scheduled execution, a convi...",English,29.0,100.0,211.5,1.785714,49.0,5.218144,0.682302,13.0,570.5,1328.0,-7.964286,36.0
270,Supercell,2023-03-17,Action,Good-hearted teenager William always lived in ...,English,56.0,100.0,910.5,-1.285714,66.0,4.333072,1.611194,2.0,1162.0,1343.0,-5.250000,59.0
271,My Otaku Girlfriend,2023-09-28,Drama,"""Get ready for an unforgettable experience whe...",Spanish,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
